# dukascopy-data-downloader (Python)

## 项目目标

> 从 [Dukascopy Datafeed](https://www.dukascopy.com/datafeed) 下载历史 K 线原始 `.bi5` 文件（保持原始格式，不解码）
> 布局：`{SYMBOL}/{YYYY}[/{MM}[/{DD}]]/{PRICE}_{candles_min_1|hour_1|day_1}.bi5`，
> 本地严格镜像官方：`{dir}/dukascopy/{SYMBOL}/{YYYY}[/{MM}[/{DD}]]/{PRICE}_{candles_*}.bi5`（URL path原样拼接，旧`{symbol}/{tf}/{symbol}-{tf}-*`保留不动）

## 配置

In [ ]:
# 配置（对标 config.rs）
symbol = ["EURUSD"]  # 必填
timeframe = ["D1"]  # 必填
price_type = "BID"  # 可选
start = "2025-01-01"  # 可选，缺省 2020-01-01
end = None  # 可选，缺省昨天
dir = "data"  # 可选
workers = 30  # 对标 Semaphore(30)

print(symbol, timeframe, price_type, start, end, dir, workers)


In [ ]:
# 效验 + end 处理（同格）
from datetime import date, timedelta

assert symbol, "symbol must not be empty"
assert all(t in ("M1", "H1", "D1", "1m", "1h", "1d") for t in timeframe)
assert price_type in ("BID", "ASK")
start = start or "2020-01-01"
yesterday = (date.today() - timedelta(days=1)).isoformat()
end = min(end, yesterday) if end else yesterday
assert start <= end, f"start {start} is after end {end}"
print("validate ok:", start, end)


## 构建下载任务

In [ ]:
# 组任务（对标 01_build_tasks：timeframe→周期→URL）
BASE = "https://www.dukascopy.com/datafeed"

def tf_part(tf):
    return {"M1": "candles_min_1", "1m": "candles_min_1",
            "H1": "candles_hour_1", "1h": "candles_hour_1"}.get(tf, "candles_day_1")

def months(s, e):
    d = date.fromisoformat(s).replace(day=1)
    out = []
    while d.isoformat() <= e:
        out.append(d)
        d = (d.replace(day=28) + timedelta(days=4)).replace(day=1)
    return out

tasks = []
for sym in symbol:
    for tf in timeframe:
        part = tf_part(tf)
        if part == 'candles_day_1':
            for y in range(int(start[:4]), int(end[:4]) + 1):
                fn = f"{price_type}_{part}.bi5"
                tasks.append((f"{BASE}/{sym}/{y}/{fn}",
                              f"dukascopy/{sym}/{y}/{fn}"))
        elif part == 'candles_hour_1':
            for d in months(start, end):
                fn = f"{price_type}_{part}.bi5"
                tasks.append((f"{BASE}/{sym}/{d:%Y/%m}/{fn}",
                              f"dukascopy/{sym}/{d:%Y/%m}/{fn}"))
        else:
            d0, d1 = date.fromisoformat(start), date.fromisoformat(end)
            while d0 <= d1:
                fn = f"{price_type}_{part}.bi5"
                tasks.append((f"{BASE}/{sym}/{d0:%Y/%m/%d}/{fn}",
                              f"dukascopy/{sym}/{d0:%Y/%m/%d}/{fn}"))
                d0 += timedelta(days=1)

print(len(tasks), tasks[0])


## 扫描本地去重

In [ ]:
# 扫描本地（相对 .bi5 集合过滤）
from pathlib import Path

local = set()
root = Path(dir)
if root.exists():
    for f in root.rglob("*"):
        if f.suffix == ".tmp":
            f.unlink(missing_ok=True)
        elif f.suffix == ".bi5":
            local.add(str(f.relative_to(root)))

todo = [(u, r) for u, r in tasks if r not in local]
print(f"total={len(tasks)} local={len(local)} todo={len(todo)}")


## 执行下载

In [ ]:
# 下载（有界并发 + 流式落盘 + 404 容忍）
from concurrent.futures import ThreadPoolExecutor
import time
import requests

ok = not_found = failed = 0

def fetch(task):
    url, rel = task
    dest = Path(dir) / rel
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 404:
                return "not_found"
            r.raise_for_status()
            dest.parent.mkdir(parents=True, exist_ok=True)
            tmp = dest.with_suffix(".bi5.tmp")
            tmp.write_bytes(r.content)
            tmp.rename(dest)
            return "ok"
        except Exception:
            time.sleep(2 ** attempt)
    return "failed"

with ThreadPoolExecutor(max_workers=workers) as ex:
    for i, st in enumerate(ex.map(fetch, todo), 1):
        ok += st == "ok"
        not_found += st == "not_found"
        failed += st == "failed"
        if i % 20 == 0:
            print(f"{i}/{len(todo)} ok={ok} not_found={not_found} failed={failed}")

print(f"done: ok={ok} not_found={not_found} failed={failed}")


## 汇总

In [ ]:
# 汇总
assert failed == 0, f"{failed}/{len(todo)} failed"
disk = [f for f in Path(dir).rglob("*.bi5")]
print(f"tasks={len(tasks)} todo={len(todo)} ok={ok} disk_bi5={len(disk)}")
